In [1]:
!pip install torchtext==0.4.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 1.9 MB/s eta 0:00:00


In [2]:
from torchtext import data, datasets
import torch
# For tokenization
TEXT = data.Field(tokenize='spacy',
                  tokenizer_language='en_core_web_sm',
                  include_lengths = True)
# For multi-class classification labels
LABEL = data.LabelField()
# Load the TREC dataset
train_data, test_data = datasets.TREC.splits(TEXT, LABEL, fine_grained=False)

downloading train_5500.label


train_5500.label: 100%|██████████| 336k/336k [00:00<00:00, 2.27MB/s]


downloading TREC_10.label


TREC_10.label: 100%|██████████| 23.4k/23.4k [00:00<00:00, 1.69MB/s]


In [3]:
TEXT.build_vocab(train_data,
                max_size=25000,
                vectors="glove.6B.100d",
                unk_init=torch.Tensor.normal_)
LABEL.build_vocab(train_data)

.vector_cache/glove.6B.zip: 862MB [02:39, 5.39MB/s]                           
100%|█████████▉| 399999/400000 [00:27<00:00, 14336.06it/s]


In [4]:
print(f"Number of unique tokens in TEXT vocabulary: {len(TEXT.vocab)}")

Number of unique tokens in TEXT vocabulary: 9330


In [5]:
oov_words = []
for example in train_data.examples:
    for token in example.text:
        if token not in TEXT.vocab.stoi:
            oov_words.append(token)

unique_oov_words = list(set(oov_words))
print(f"Number of out-of-vocabulary words: {len(unique_oov_words)}")
print(f"First 20 out-of-vocabulary words: {unique_oov_words[:20]}")

Number of out-of-vocabulary words: 0
First 20 out-of-vocabulary words: []


In [6]:
oov_counts_by_topic = {}

for example in train_data.examples:
    label = example.label
    tokens = example.text

    if label not in oov_counts_by_topic:
        oov_counts_by_topic[label] = 0

    for token in tokens:
        if token not in TEXT.vocab.stoi:
            oov_counts_by_topic[label] += 1

print("Number of out-of-vocabulary words per topic category:")
for topic, count in oov_counts_by_topic.items():
    print(f"Topic: {topic}, OOV Count: {count}")

Number of out-of-vocabulary words per topic category:
Topic: DESC, OOV Count: 0
Topic: ENTY, OOV Count: 0
Topic: ABBR, OOV Count: 0
Topic: HUM, OOV Count: 0
Topic: NUM, OOV Count: 0
Topic: LOC, OOV Count: 0


In [7]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stopwords = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [8]:
word_counts_by_topic = {}

for example in train_data.examples:
    label = example.label
    tokens = example.text

    if label not in word_counts_by_topic:
        word_counts_by_topic[label] = {}

    for token in tokens:
        lower_token = token.lower()
        if lower_token not in stopwords:
            word_counts_by_topic[label][lower_token] = word_counts_by_topic[label].get(lower_token, 0) + 1

In [9]:
from collections import Counter

top_20_words_by_topic = {}

for topic, word_counts in word_counts_by_topic.items():
    counter = Counter(word_counts)
    top_20_words = counter.most_common(20)
    top_20_words_by_topic[topic] = top_20_words

In [10]:
for topic, top_words in top_20_words_by_topic.items():
    print(f"Topic: {topic}")
    for word, count in top_words:
        print(f"  {word}: {count}")

Topic: DESC
  ?: 1151
  `: 243
  '': 112
  's: 81
  ,: 79
  mean: 62
  origin: 54
  -: 46
  get: 37
  ': 34
  name: 32
  difference: 32
  word: 30
  find: 27
  come: 22
  work: 18
  causes: 17
  term: 16
  make: 16
  meaning: 16
Topic: ENTY
  ?: 1216
  's: 206
  `: 193
  ,: 173
  name: 119
  -: 106
  '': 96
  fear: 66
  first: 50
  .: 43
  kind: 43
  called: 41
  used: 38
  world: 37
  film: 36
  color: 33
  book: 27
  best: 27
  made: 27
  war: 25
Topic: ABBR
  ?: 86
  stand: 41
  abbreviation: 16
  `: 16
  ,: 9
  mean: 9
  '': 6
  national: 5
  bureau: 5
  investigation: 5
  's: 4
  acronym: 4
  used: 3
  ': 3
  cnn: 3
  name: 3
  abbreviated: 3
  letters: 3
  ioc: 3
  e: 3
Topic: HUM
  ?: 1181
  `: 287
  's: 238
  ,: 151
  name: 144
  '': 142
  -: 126
  first: 82
  president: 65
  .: 49
  company: 44
  u.s.: 40
  wrote: 38
  :: 37
  world: 37
  invented: 33
  famous: 32
  ': 28
  tv: 27
  two: 26
Topic: NUM
  ?: 892
  many: 323
  's: 90
  ,: 72
  year: 68
  much: 57
  long: 56
  `: 

In [15]:
import numpy as np

word_list = TEXT.vocab.itos
word_embeddings = TEXT.vocab.vectors

word_array = np.array(word_list, dtype=object)
embeddings_array = word_embeddings.numpy()

word_list_filename = "word_list.npy"
np.save('word_list.npy', word_array)
np.save('word_embeddings.npy', embeddings_array)